# Jury LLM: Multi-Model & Human-in-the-Loop Evaluation System

This notebook serves as the interactive frontend for the Jury System.

In [3]:
import sys
import os
sys.path.append('../')

from dotenv import load_dotenv
load_dotenv('../.env')

import ipywidgets as widgets
from IPython.display import display, Markdown, clear_output
from src.graph import app
from src.utils import parse_json_output
from src.llm_provider import LLMProvider
from src.agents import QUALIFICATION_PROMPT
from src.qualification import QualificationFlow
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv(), override=True)

llm = LLMProvider()
flow = QualificationFlow()

INFO:src.llm_provider:Using DashScope configuration.
INFO:src.llm_provider:Using DashScope configuration.
INFO:src.llm_provider:Using DashScope configuration.


## Step 1: Input Target Text
Please paste the LLM generated text you want to evaluate below. This will be stored globally for the entire session.

In [5]:
import os
# Fix for 502 Bad Gateway / Proxy issues with Gradio
os.environ['no_proxy'] = 'localhost,127.0.0.1'
if 'http_proxy' in os.environ: del os.environ['http_proxy']
if 'https_proxy' in os.environ: del os.environ['https_proxy']

import gradio as gr

# --- Global Variables ---
TARGET_TEXT = ""
EVALUATION_PURPOSE = "General Assessment"
HUMAN_COMPETENCY_SCORE = 0
QUALIFICATION_HISTORY = []
MAX_ROUNDS = 3  # Checkpoint had 3 rounds
CURRENT_ROUND = 0

def save_step1_inputs(text, purpose):
    global TARGET_TEXT, EVALUATION_PURPOSE
    
    if not text.strip():
        return "❌ Error: Target Text cannot be empty."
    
    TARGET_TEXT = text
    EVALUATION_PURPOSE = purpose
    flow.set_target_text(text, purpose)
    
    return f'''✅ Saved Successfully!
    
    Target Text Length: {len(text)} chars
    Evaluation Purpose: {purpose}
    
    You can now stop this cell and run Step 2.'''

with gr.Blocks() as step1_demo:
    gr.Markdown("## Step 1: Input Target Text")
    gr.Markdown("Please paste the LLM generated text you want to evaluate below and specify your evaluation goal.")
    
    txt_input = gr.Textbox(
        label="Target Text", 
        lines=10, 
        placeholder="Paste the LLM generated text here..."
    )
    
    purpose_input = gr.Textbox(
        label="Evaluation Purpose",
        value="General Assessment",
        lines=2,
        placeholder="e.g. Verify legal accuracy, Check for creative writing style, etc."
    )
    
    submit_btn = gr.Button("Confirm & Save Text", variant="primary")
    output_msg = gr.Markdown()
    
    submit_btn.click(
        fn=save_step1_inputs, 
        inputs=[txt_input, purpose_input], 
        outputs=output_msg
    )

print("Launching Step 1 Interface...")
try:
    step1_demo.launch(height=600, inline=True, quiet=False)
except Exception as e:
    print(f"Error launching Gradio: {e}")


INFO:httpx:HTTP Request: GET http://127.0.0.1:7860/gradio_api/startup-events "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD http://127.0.0.1:7860/ "HTTP/1.1 200 OK"


Launching Step 1 Interface...
* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


## Step 2: Qualification Assessment
The AI will now analyze the text you provided in Step 1 and assess your qualification to evaluate it.

In [9]:
import os
# Fix for 502 Bad Gateway / Proxy issues with Gradio
os.environ['no_proxy'] = 'localhost,127.0.0.1'
if 'http_proxy' in os.environ: del os.environ['http_proxy']
if 'https_proxy' in os.environ: del os.environ['https_proxy']

import gradio as gr

with gr.Blocks() as step2_demo:
    gr.Markdown("## Step 2: Qualification Assessment")
    gr.Markdown("Click **Start Assessment** to begin.")
    
    with gr.Row():
        max_rounds_input = gr.Slider(minimum=3, maximum=50, value=15, step=1, label="Max Rounds")      
        start_btn = gr.Button("Start Assessment", variant="primary")
        clear_btn = gr.ClearButton(value="Reset")
    
    # Passed without type="messages" to be safe for older Gradio versions.
    chatbot = gr.Chatbot(height=500, label="Interview Chat")
    msg = gr.Textbox(label="Your Answer", placeholder="Type here...", interactive=False)
    
    start_btn.click(flow.on_start_interview, inputs=[max_rounds_input], outputs=[chatbot, msg])
    msg.submit(flow.on_user_reply, [msg, chatbot], [chatbot, msg])
    clear_btn.click(lambda: None, None, chatbot, queue=False)

print("Launching Step 2 Interface...")
try:
    step2_demo.launch(height=600, inline=True, quiet=False)
except Exception as e:
    print(f"Error launching Gradio: {e}")


Launching Step 2 Interface...


INFO:httpx:HTTP Request: GET http://127.0.0.1:7861/gradio_api/startup-events "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD http://127.0.0.1:7861/ "HTTP/1.1 200 OK"


* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


## Step 2.5: Evaluation Criteria (Rubrics)
Define the evaluation criteria that will be used by all judges in Step 3.

You can either:
1. **Provide your own criteria** (paste into the text box)
2. **Generate criteria using AI** (leave text box empty and click Generate)

In [12]:
import os
# Fix for 502 Bad Gateway / Proxy issues with Gradio
os.environ['no_proxy'] = 'localhost,127.0.0.1'
if 'http_proxy' in os.environ: del os.environ['http_proxy']
if 'https_proxy' in os.environ: del os.environ['https_proxy']

import gradio as gr
from src.rubrics import RubricsFlow

# --- Global Variable for Rubrics ---
EVALUATION_RUBRICS = ""

# Initialize rubrics flow
rubrics_flow = RubricsFlow()
rubrics_flow.set_context(TARGET_TEXT, EVALUATION_PURPOSE)

def on_use_manual_rubrics(manual_input):
    """User provides their own rubrics."""
    global EVALUATION_RUBRICS
    
    if not manual_input or not manual_input.strip():
        return "❌ Error: Please paste your evaluation criteria.", "", gr.update(interactive=False)
    
    EVALUATION_RUBRICS = manual_input.strip()
    rubrics_flow.store_rubrics(EVALUATION_RUBRICS)
    
    return (
        f"✅ Your criteria have been saved! ({len(EVALUATION_RUBRICS)} characters)",
        EVALUATION_RUBRICS,
        gr.update(interactive=True)
    )

def on_generate_rubrics(manual_input):
    """Generate rubrics using AI."""
    global EVALUATION_RUBRICS
    
    # Only generate if manual input is empty
    if manual_input and manual_input.strip():
        return (
            "⚠️ Manual criteria detected. Please clear the manual input first or click 'Use Manual Criteria'.",
            manual_input,
            gr.update(interactive=False)
        )
    
    if not TARGET_TEXT:
        return "❌ Error: No target text found. Please complete Step 1 first.", "", gr.update(interactive=False)
    
    # Generate rubrics
    generated = rubrics_flow.generate_rubrics()
    
    if generated.startswith("❌"):
        return generated, "", gr.update(interactive=False)
    
    return (
        "✅ Criteria generated! Please review and edit if needed, then click 'Confirm and Continue'.",
        generated,
        gr.update(interactive=True)
    )

def on_confirm_rubrics(edited_rubrics):
    """Confirm and store the final rubrics."""
    global EVALUATION_RUBRICS
    
    if not edited_rubrics or not edited_rubrics.strip():
        return "❌ Error: Cannot save empty criteria."
    
    EVALUATION_RUBRICS = edited_rubrics.strip()
    rubrics_flow.store_rubrics(EVALUATION_RUBRICS)
    
    return f'''✅ Evaluation Criteria Confirmed!
    
Criteria Length: {len(EVALUATION_RUBRICS)} characters

**Preview:**
{EVALUATION_RUBRICS[:500]}...

You can now stop this cell and proceed to Step 3.'''

with gr.Blocks() as step25_demo:
    gr.Markdown("## Step 2.5: Evaluation Criteria (Rubrics)")
    gr.Markdown("""Define the criteria that all judges will use in Step 3. 
    You can provide your own criteria or generate them using AI.""")
    
    with gr.Row():
        with gr.Column():
            gr.Markdown("### Option 1: Manual Input (Priority)")
            manual_input = gr.Textbox(
                label="Your Evaluation Criteria",
                lines=8,
                placeholder="Paste your evaluation criteria here (in markdown format)..."
            )
            use_manual_btn = gr.Button("Use Manual Criteria", variant="primary")
        
        with gr.Column():
            gr.Markdown("### Option 2: AI Generation")
            gr.Markdown("*Leave manual input empty to use AI generation.*")
            generate_btn = gr.Button("Generate Evaluation Criteria", variant="secondary")
    
    status_msg = gr.Markdown()
    
    gr.Markdown("### Review & Edit")
    edit_area = gr.Textbox(
        label="Final Criteria (Editable)",
        lines=15,
        placeholder="Your criteria will appear here for review and editing...",
        interactive=False
    )
    
    confirm_btn = gr.Button("Confirm and Continue", variant="primary", interactive=False)
    final_msg = gr.Markdown()
    
    # Event handlers
    use_manual_btn.click(
        fn=on_use_manual_rubrics,
        inputs=[manual_input],
        outputs=[status_msg, edit_area, confirm_btn]
    )
    
    generate_btn.click(
        fn=on_generate_rubrics,
        inputs=[manual_input],
        outputs=[status_msg, edit_area, confirm_btn]
    )
    
    confirm_btn.click(
        fn=on_confirm_rubrics,
        inputs=[edit_area],
        outputs=[final_msg]
    )

print("Launching Step 2.5 Interface...")
try:
    step25_demo.launch(height=700, inline=True, quiet=False)
except Exception as e:
    print(f"Error launching Gradio: {e}")


INFO:src.llm_provider:Using DashScope configuration.


Launching Step 2.5 Interface...


INFO:httpx:HTTP Request: GET http://127.0.0.1:7862/gradio_api/startup-events "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD http://127.0.0.1:7862/ "HTTP/1.1 200 OK"


* Running on local URL:  http://127.0.0.1:7862
* To create a public link, set `share=True` in `launch()`.


## Step 3: Jury Evaluation
Now that your qualification is established, we proceed to the jury evaluation.

In [14]:
# Widgets for Jury Step
human_score_input = widgets.FloatSlider(value=80, min=0, max=100, description='Your Score:')
human_reason_input = widgets.Textarea(description='Reason:', placeholder='Why did you give this score?')
start_jury_btn = widgets.Button(description='Start Jury Evaluation', button_style='success')
jury_output_area = widgets.Output()

display(human_score_input, human_reason_input, start_jury_btn, jury_output_area)

thread = {"configurable": {"thread_id": "1"}}

def on_start_jury(b):
    with jury_output_area:
        clear_output()

        # Sync Score from Flow
        global HUMAN_COMPETENCY_SCORE
        HUMAN_COMPETENCY_SCORE = flow.get_score()

        if not TARGET_TEXT:
            print("Error: No Target Text found. Please complete Step 1.")
            return
            
        print(f"Initializing Jury with Human Competency Score: {HUMAN_COMPETENCY_SCORE}...")
        
        # Map Competency Score (0-100) to Weight (0.5-1.5)
        human_weight = 0.5 + (HUMAN_COMPETENCY_SCORE / 100.0)
        
        initial_state = {
            "topic": TARGET_TEXT,
            "human_bio": "Assessed via Qualification Exam",
            "human_score": human_score_input.value,
            "human_reason": human_reason_input.value,
            "human_weight": human_weight,
            "evaluation_rubrics": EVALUATION_RUBRICS,
            "model_outputs": {},
            "debate_logs": [],
            "votes": {}
        }
        
        for event in app.stream(initial_state, thread, stream_mode="values"):
            if 'debate_logs' in event and event['debate_logs']:
                print("--- Debate Log ---")
                for log in event['debate_logs']:
                    print(log)
            if 'anonymized_reasons' in event:
                print("Ready for Voting...")
                
        # Check state after interruption
        state_snapshot = app.get_state(thread)
        if state_snapshot.next:
            print("System paused for Human Vote.")
            show_voting_ui(state_snapshot.values)

start_jury_btn.on_click(on_start_jury)

FloatSlider(value=80.0, description='Your Score:')

Textarea(value='', description='Reason:', placeholder='Why did you give this score?')

Button(button_style='success', description='Start Jury Evaluation', style=ButtonStyle())

Output()

## Step 4: Voting Interface

In [16]:
vote_widget = widgets.RadioButtons(options=[], description='Best Reason:')
submit_vote_btn = widgets.Button(description='Submit Vote', button_style='info')
vote_output = widgets.Output()

def show_voting_ui(state_values):
    options = state_values['anonymized_reasons']
    display_options = []
    for opt_id, text in options.items():
        display_options.append((f"{opt_id}: {text[:100]}...", opt_id))
    
    vote_widget.options = display_options
    display(vote_widget, submit_vote_btn, vote_output)

def on_vote_submit(b):
    with vote_output:
        selected_option = vote_widget.value
        print(f"You voted for: {selected_option}")
        
        current_votes = app.get_state(thread).values.get('votes', {})
        current_votes['human'] = selected_option
        
        app.update_state(thread, {"votes": current_votes})
        
        print("Resuming execution...")
        for event in app.stream(None, thread, stream_mode="values"):
            if 'final_verdict' in event:
                display(Markdown("### Final Verdict"))
                display(Markdown(event['final_verdict']))

submit_vote_btn.on_click(on_vote_submit)